In [ ]:
#| hide

from nbdev import show_doc

DEBUG:fhemb.config.settings:Loading environment from /Users/radned/.config/fhemb/.env.paths
DEBUG:fhemb.config.settings:Loading environment from /Users/radned/.config/fhemb/.env.db


In [ ]:
from fhemb.embedding import Embedding 
from fhemb.utils.factories import ConcreteEmbFactory, wfactory

# `Embedding` API
> The `Embedding` class encapsulates an individual feature vector and exposes analysis and similarity methods tailored to thermal‑face representations.

#### The Method Resolution Order (MRO) 
>the exact linear path Python follows when looking up attributes or when super() is used.

```raw
Embedding
 └── BaseEmbedding
      └── abc.ABC
           └── ValidateSubjsMixin
                └── object
```

#### Architectural responsibilities

- **Final product**: hold per‑subject feature×time matrices (`features_tint`) plus minimal provenance (feature names, time interval, sampling rate, optional binedges/percentile info).

- **Interchange format**: every higher‑level transform (*PCA*, *concatenation*, *wavelet decomposition*, *reconstruction*) consumes and returns `Embedding` objects — embeddings are both outputs and inputs for further analysis.

- **Deterministic primitives**: provide stable, deterministic operations for slicing, projecting, and inspecting matrices so analyses compose reliably.

- **Crucial point**: the system’s main goal is to analyze `Embedding` objects; all pipelines ultimately produce embeddings for downstream analysis.

In [ ]:
show_doc(Embedding, name="The Constructor", title_level=2)

---

## The Constructor

```python

def Embedding(
    time_interval:Tuple[int, int], # Start and end time points for the embedding.
    features_tint:Dict[Subjs, np.matrix], # matrix features x time
    features:List[Union[int, str]]=None, # list of features to use
    subjects:List[str]=None, # List of subjects. If None, inferred from features_tint keys.
    temperature_threshold:float=30, # Threshold for temperature-based filtering.
    percentile_intv:NoneType=None, # Percentile interval for feature processing.
    binedges:Dict[Subjs, np.matrix['binintervals-2bounds']]=None, # Bin edges for histogram-based features.
    periods:List[Tuple[int, int]]='All', # Periods to include in the embedding.
    frequencies:List[Tuple[int, int]]='All', # list of periods to use for the embedding
    kwargs:VAR_KEYWORD
):


```

*A concrete embedding class that adds wavelet-specific behavior.*

This class extends BaseEmbedding to provide wavelet decomposition capabilities
for signal processing and analysis.

## Creating Another Embedding

In [ ]:
show_doc(Embedding.project, name='project', title_level=4)

---

#### project

```python

def project(
    subjs:Union[List[int], str]='all', # List of subject indices or `'all'`.
    features:Optional[List[str]]=None, # Feature names to project.
    heatmap:Optional[np.ndarray]=None, # Heatmap data to project.
    fvector:Optional[np.ndarray]=None, # Feature vector to project.
    fbands:Optional[List[int]]=None, # Frequency bands to project.
    factory:Optional['ConcreteEmbFactory']=None, # Wavelet factory for spectral decomposition.
)->'Embedding': # The projected embedding.


```

*Project a new embedding for specified subjects and features.*

In [ ]:
show_doc(Embedding.bbootstrap, name='bbootstrap', title_level=4)

---

#### bbootstrap

```python

def bbootstrap(
    subjs:Union[List[int], str]=None, # Subjects to bootstrap.
    err_ts:np.matrix=None, # Error time series.
    block_length:int=None, # in seconds
    max_level:int=8, # Maximum level of the wavelet decomposition.
    rseed:int=None, # Random seed.
)->'Embedding': # Bootstrap resampled embedding.


```

*Perform block bootstrapping.*
If you want to preserve the subsignals that live at wavelet level L, choose max_level = L (or set block_length so it matches the period computed from the formula).

In [ ]:
#| hide
from fhemb.utils.factories import ConcreteEmbFactory

In [ ]:
#| hide
# Fix incorrect annotation in source code
Embedding.create_wdecomposition.__annotations__['factory'] = ConcreteEmbFactory

In [ ]:
show_doc(Embedding.create_wdecomposition, name='create_wdecomposition', title_level=4)

## Wavelet Decomposition

In [ ]:
show_doc(wfactory,  title_level=4)

---

#### wfactory

```python

def wfactory(
    wtype:str='db4', # The type of wavelet to use (default is "db4"). Examples include "db2", "sym2", "coif1", etc.
    level:int=8, # The level of wavelet decomposition (default is 8). Higher levels capture lower frequency components.
)->ConcreteEmbFactory: # An instance of ConcreteEmbFactory configured with the specified wavelet type and level.


```

*Factory function to create a ConcreteEmbFactory with specified wavelet type and level.*

In [ ]:
show_doc(ConcreteEmbFactory, name="The constructor", title_level=5)

/Users/radned/.pyenv/versions/p311.fhemb/lib/python3.11/site-packages/fastcore/docscrape.py:259: UserWarning: potentially wrong underline length... 
Parameters: 
---------- in 
Parameters:
----------...
  else: warn(msg)


---

##### The constructor

```python

def ConcreteEmbFactory(
    level, wtype
):


```

*A concrete implementation of the DecompositionFactory abstract class.*
Provides methods for creating wavelet decomposition dictionaries and reconstruction embeddings.

In [ ]:
LEVEL = 8  # Level of wavelet decomposition, which determines the frequency resolution 

::: {.callout-note}
`LEVEL` 
The number of successive wavelet decomposition stages applied to the input signal. Each level produces one detail subband, so a `LEVEL=N` decomposition yields `N` detail bands plus a final approximation band.
:::

#### Wavelet transformations
> The table below documents the wavelet configurations used by the embedding pipeline.  

::: {.callout collapse="true" title="Wavelet models (spectral decomposition stage) — API summary"}
| `wtype` value | support length formula | family       | description                                                                 | source                                                                               |
|:-------------|:------------------------|:-------------|:-----------------------------------------------------------------------------|:-------------------------------------------------------------------------------------|
| `db<n>`        | $2n$                     | Daubechies   | Orthogonal wavelets with n vanishing moments; compact support                | Daubechies, *Ten Lectures on Wavelets* (1992)                                        |
| `sym<n>`       | $2n$                     | Symlets      | Nearly symmetric orthogonal wavelets with n vanishing moments                | Daubechies, *Ten Lectures on Wavelets* (1992)                                        |
| `coif<n>`      | $6n$                     | Coiflets     | Orthogonal wavelets with n vanishing moments for the scaling and wavelet functions; balanced scaling | Daubechies & Coifman, *Ten Lectures on Wavelets* (1992)                              |
| `bior1.3`      | $6$                      | Biorthogonal | Linear‑phase biorthogonal pair; symmetric reconstruction filters             | Cohen–Daubechies–Feauveau, *Biorthogonal Bases of Compactly Supported Wavelets* (1992) |
:::

::: {.callout collapse="true" title="Parameter effects — API summary"}
| wavelet family | parameterization |  smoothness (regularity) | effect of increasing n |
|:---------------|:-----------------|:-------------------------|:------------------------|
| Daubechies `db<n>` | $n$ vanishing moments | increases (Hölder regularity grows with $n$) | longer support, smoother wavelet, better frequency localization |
| Symlets `sym<n>`   | $n$ vanishing moments | increases (similar to `db<n>`) | same as Daubechies but with improved symmetry |
| Coiflets `coif<n>` | $n$ vanishing moments for scaling and wavelet functions | increases (coiflets are highly regular for large $n$) | very long support, high smoothness, balanced scaling function |
| Biorthogonal `bior1.3` | fixed pair | fixed (depends on filter design) | no parametric effect; smoothness determined by filter pair |
:::

::: {.callout collapse="true" title="Wavelet family comparison — API summary"}
| family       | orthogonality | symmetry | vanishing moments | typical use cases |
|:-------------|:--------------|:---------|:------------------|:------------------|
| Daubechies   | orthogonal    | low      | $n$                 | general‑purpose analysis; compact support; sparse representations |
| Symlets      | orthogonal    | improved | $n$                 | applications requiring near‑symmetry; reduced phase distortion |
| Coiflets     | orthogonal    | moderate | $n$ for scaling and wavelet functions | high‑regularity tasks; balanced scaling; derivative estimation |
| Biorthogonal | biorthogonal  | high     | varies by pair    | linear‑phase filtering; image processing; symmetric reconstruction |
:::


#### Predefined wavelet factories

In [ ]:
wfactories = dict(
    db2=ConcreteEmbFactory(
        wtype="db2", 
        level=LEVEL
    ),  # Daubechies — support length = 4

    db4=ConcreteEmbFactory(
        wtype="db4", 
        level=LEVEL
    ),  # Daubechies — support length = 8

    db10=ConcreteEmbFactory(
        wtype="db10", 
        level=LEVEL
    ),  # Daubechies — support length = 20

    sym2=ConcreteEmbFactory(
        wtype="sym2", 
        level=LEVEL
    ),  # Symlet — support length = 4

    sym10=ConcreteEmbFactory(
        wtype="sym10", 
        level=LEVEL
    ),  # Symlet — support length = 20

    bior13=ConcreteEmbFactory(
        wtype="bior1.3", 
        level=LEVEL
    ),  # Biorthogonal (CDF 1.3) — support length = 6

    coif1=ConcreteEmbFactory(
        wtype="coif1", 
        level=LEVEL
    ),  # Coiflet — support length = 6

    coif5=ConcreteEmbFactory(
        wtype="coif5", 
        level=LEVEL
    ),  # Coiflet — support length = 30
)


## Utilities

### Inherited from BaseEmbedding

In [ ]:
show_doc(Embedding.get_ts_subjs, name='get_ts_subjs', title_level=4)

---

#### get_ts_subjs

```python

def get_ts_subjs(
    args:Union[int, str, list, tuple]=None, # Features of interest (e.g., 'median', 'mean', 'std'). If None, uses the first feature.
)->Union[pd.DataFrame, Dict['feature', pd.DataFrame]]: # DataFrame with subjects as columns (for a single feature) or a dict mapping feature -> DataFrame.


```

*Return time series data for subjects.*

### Plotting / visualization helpers

In [ ]:
show_doc(Embedding.plot_signal, name='plot_signal', title_level=4)

---

#### plot_signal

```python

def plot_signal(
    subjs:List[str], # Subjects to plot.
    feature:str=None, # Feature index to plot. If None and multiple features exist, an error is raised.
    title:str='', # Title suffix for the plot.
    label:str='', # Label for the plot.
    linestyle:str='-', # Line style.
    show:bool=True, # Whether to call `plt.show()`.
): # (fig, axes) of the created matplotlib figure.


```

*Plot the signals for given subjects.*

In [ ]:
show_doc(Embedding.plot_temperaturebars, name='plot_temperaturebars', title_level=4)

---

#### plot_temperaturebars

```python

def plot_temperaturebars(
    subjs:List[int]='all', # Subjects to plot.
    frms:List[int]=None, # Frames to display.
    sharex:bool=True, # Share x-axis between plots.
    sharey:bool=True, # Share y-axis between plots.
    centralized:bool=False, # Whether to centralize the values around the mean.
    kwargs:VAR_KEYWORD
):


```

*Plot temperature bar charts for selected subjects and frames.*

In [ ]:
show_doc(Embedding.plot_temperaturehist, name='plot_temperaturehist', title_level=4)

---

#### plot_temperaturehist

```python

def plot_temperaturehist(
    subjs:List[int]='all', # Subjects to plot.
    frms:List[int]=None, # Frames to include.
    sharex:bool=True, # Share x-axis between plots.
    sharey:bool=True, # Share y-axis between plots.
    kwargs:VAR_KEYWORD
):


```

*Plot heatmap histograms for subjects and specified frames.*

In [ ]:
show_doc(Embedding.plot_intvlhist, name='plot_intvlhist', title_level=4)

---

#### plot_intvlhist

```python

def plot_intvlhist(
    subjs, # Subjects to plot.
    intvl:tuple=(0, 10), # Time interval (start, end) in frames.
    sharex:bool=True, # Share x-axis between plots.
    sharey:bool=True, # Share y-axis between plots.
    feature:str | None=None, # Feature to plot.
    kwargs:VAR_KEYWORD
):


```

*Plot histograms of temperature bins for a given interval.*

In [ ]:
show_doc(Embedding.visualize_eigenfaces, name='visualize_eigenfaces', title_level=4)

---

#### visualize_eigenfaces

```python

def visualize_eigenfaces(
    subjs:list | None=None, # Subjects to visualize. If None, uses all available subjects.
    components:list[int] | None=None, # Eigenface indices to display.
    cmap:str='hot', # Colormap for visualization.
    kwargs:VAR_KEYWORD
)->Dict[str, List[plt.Figure]]: # Mapping from subject to list of figures.


```

*Visualize eigenfaces for a list of subjects.*

### Other useful properties *(readonly)*

- **`subjects`** 

- **`features`**

- **`periods`** 

- **`frequencies`** 

- **`binedges`**  

- **`features_tint`** 

- **`sr = 25`** *(sampling rate)*